In [4]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
import numpy as np
import pandas as pd
from sqlalchemy import create_engine

# --- 1. Setup ---
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from algogators_wrisk import config, features, analysis

# Connection
conn_str = f"postgresql+psycopg://{os.environ['DB_USER']}:{os.environ['DB_PASSWORD']}@{os.environ['DB_HOST']}:5432/{os.environ['DB_NAME']}"
engine = create_engine(conn_str)

def fetch_data(symbols, start, end):
    price_series = []
    for symbol in symbols:
        query = f"SELECT time, close FROM {config.DB_SCHEMA}.{config.PRICES_TABLE} WHERE symbol = '{symbol}' AND time BETWEEN '{start}' AND '{end}' ORDER BY time ASC"
        df = pd.read_sql(query, engine)
        if not df.empty:
            df['time'] = pd.to_datetime(df['time']).dt.floor('D')
            s = df.set_index('time')['close'].rename(symbol)
            price_series.append(s[~s.index.duplicated(keep='last')])
    return pd.concat(price_series, axis=1).sort_index().ffill().dropna()

# --- 2. Execution ---
print("--- Loading Data ---")
# Fetch everything at once
all_data = fetch_data(config.UNIVERSE, config.START_DATE, config.END_DATE)

# Separate Universe prices from Macro Treasuries
prices = all_data.drop(columns=['ZT', 'ZF'], errors='ignore')
df_macro = all_data[['ZT', 'ZF']].rename(columns={'ZT': 'treas_2y_close', 'ZF': 'treas_5y_close'})

print(f"✅ Success! Data loaded for: {list(prices.columns)}")

# --- 3. Run Wasserstein Math ---
returns = np.log(prices).diff().dropna()

# Note: Calling compute_wasserstein_shift_index from your features.py
w_index = features.compute_wasserstein_shift_index(returns, window=config.RV_PAST_WINDOW)

# Build Analysis Panel
panel = analysis.build_core_panel(
    prices=prices,
    returns=returns,
    w_index=w_index,
    rv_past_window=config.RV_PAST_WINDOW,
    rv_future_window=config.RV_FUTURE_WINDOW,
    lambda1_window=config.LAMBDA1_WINDOW
)

# Join Macro
panel = panel.join(df_macro, how='inner').dropna()

# --- 4. Final Regression ---
results = analysis.run_hac_regression(panel, y_col="rv_future", x_cols=["rv_past", "lambda1", "w_index"], lags=config.HAC_LAGS)
print("\n--- Wasserstein Risk Index Results ---")
print(results.summary())

--- Loading Data ---


AttributeError: module 'algogators_wrisk.config' has no attribute 'START_DATE'

In [4]:
# Run this to see the actual symbol names in the DB
print(pd.read_sql("SELECT DISTINCT(symbol) FROM futures_data.new_data_ohlcv_1d LIMIT 20", engine))

  symbol
0     ZT
1     ZF
2     6L
3     NG
4     6A
5     HO


In [2]:
# 0) Add autoreload so code changes pick up automatically
%load_ext autoreload
%autoreload 2

import os
import sys
import numpy as np
import pandas as pd
from sqlalchemy import create_engine

# --- 1. Setup ---
# Adjust path to find the package
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from algogators_wrisk import config, features, analysis

# Connection
conn_str = f"postgresql+psycopg://{os.environ['DB_USER']}:{os.environ['DB_PASSWORD']}@{os.environ['DB_HOST']}:5432/{os.environ['DB_NAME']}"
engine = create_engine(conn_str)

def fetch_data(symbols, start, end):
    price_series = []
    for symbol in symbols:
        # Uses the corrected config values
        query = f"SELECT time, close FROM {config.DB_SCHEMA}.{config.PRICES_TABLE} WHERE symbol = '{symbol}' AND time BETWEEN '{start}' AND '{end}' ORDER BY time ASC"
        df = pd.read_sql(query, engine)
        if not df.empty:
            df['time'] = pd.to_datetime(df['time']).dt.floor('D')
            s = df.set_index('time')['close'].rename(symbol)
            price_series.append(s[~s.index.duplicated(keep='last')])
    
    if not price_series:
        raise ValueError("No data found. Check your config.py table names and symbols.")
        
    return pd.concat(price_series, axis=1).sort_index().ffill().dropna()

# --- 2. Execution ---
print("--- Loading Data ---")
# This will now find config.START_DATE after the kernel restart
all_data = fetch_data(config.UNIVERSE, config.START_DATE, config.END_DATE)

# Separate Universe prices from Macro Treasuries
prices = all_data.drop(columns=['ZT', 'ZF'], errors='ignore')
df_macro = all_data[['ZT', 'ZF']].rename(columns={'ZT': 'treas_2y_close', 'ZF': 'treas_5y_close'})

print(f"✅ Success! Data loaded for: {list(prices.columns)}")

# --- 3. Analysis ---
returns = np.log(prices).diff().dropna()
w_index = features.compute_wasserstein_shift_index(returns, window=config.RV_PAST_WINDOW)

panel = analysis.build_core_panel(
    prices=prices,
    returns=returns,
    w_index=w_index,
    rv_past_window=config.RV_PAST_WINDOW,
    rv_future_window=config.RV_FUTURE_WINDOW,
    lambda1_window=config.LAMBDA1_WINDOW
)

panel = panel.join(df_macro, how='inner').dropna()

results = analysis.run_hac_regression(
    panel, 
    y_col="rv_future", 
    x_cols=["rv_past", "lambda1", "w_index"], 
    lags=config.HAC_LAGS
)

print("\n--- Final Results ---")
print(results.summary())

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
--- Loading Data ---
✅ Success! Data loaded for: ['NG', '6L', '6A', 'HO']


/tmp/ipykernel_17608/3907237312.py:37: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  return pd.concat(price_series, axis=1).sort_index().ffill().dropna()


TypeError: compute_wasserstein_shift_index() got an unexpected keyword argument 'window'